<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-10-production-deployment/lesson-10.8-monitoring-and-observability/notebooks/exercises-10.8.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10.8 — Monitoring & Observability
Netsetos GenAI Course v2.0 | Module 10: Production Deployment

Golden signals, Langfuse / OpenTelemetry / Prometheus / Grafana, burn-rate alerts, eval-on-prod, PII redaction, distributed tracing.


In [ ]:
print('Module 10: Production Deployment')
print('Lesson 10.8: Monitoring & Observability')


## Ex 1: 4 Golden Signals for LLM Apps


In [ ]:
print('Google SRE golden signals adapted for LLM apps:')
for signal, question, llm_metric in [
    ('Latency','How fast?','TTFT + tokens_per_sec by model'),
    ('Traffic','How much?','Requests/sec + tokens/sec by tenant'),
    ('Errors','What broke?','4xx/5xx + breaker trips + refusals'),
    ('Saturation','How full?','Queue depth, GPU util, ratelimit remaining'),
    ('+ Cost','How expensive?','$/request by tenant/feature'),
    ('+ Quality','Still good?','Eval-on-prod score, 7d moving avg'),
]:
    print(f'  {signal:14s}: {question:14s} | {llm_metric}')


## Ex 2: Observability Tool Comparison


In [ ]:
print('LLM observability tools:')
for tool, strength, best_for in [
    ('Langfuse','OSS, traces + evals + cost','Default, self-host friendly'),
    ('Helicone','Drop-in proxy, fast UX','Quick cost analytics'),
    ('LangSmith','Tight LangChain, eval-first','LangChain stacks'),
    ('Phoenix (Arize)','OSS, drift + embeddings','RAG drift, embedding stores'),
    ('Braintrust','Evals + experiments','Prompt engineering workflows'),
]:
    print(f'  {tool:18s}: {strength:32s} | Best: {best_for}')


## Ex 3: Metric Types


In [ ]:
print('Prometheus metric types:')
for mtype, when, examples in [
    ('Counter','Monotonic events','requests_total, tokens_in_total, cost_usd_total'),
    ('Gauge','Current value','queue_depth, breaker_state, gpu_mem_bytes'),
    ('Histogram','Distributions w/ quantiles','latency_ms, tokens_per_call, cost_per_call'),
    ('Summary','Pre-computed quantiles','Rare - histograms aggregate better'),
]:
    print(f'  {mtype:10s}: {when:26s} | Ex: {examples}')


## Ex 4: OpenTelemetry Trace Structure


In [ ]:
print('OTel trace: root span + children for a typical RAG call:')
for span, attrs in [
    ('http.request','route, method, tenant'),
    ('  prompt.build','template_version, token_count'),
    ('  retrieval','top_k, latency_ms, chunks_returned'),
    ('  gateway.call','provider, model, tokens_in, tokens_out'),
    ('    provider.call','TTFT_ms, cost_usd, cache_hit'),
    ('  parse','output_format, tokens_parsed'),
    ('  post.hooks','langfuse, prometheus'),
]:
    print(f'  {span:22s} | {attrs}')


## Ex 5: Alert Thresholds (Burn Rate)


In [ ]:
print('Multi-window multi-burn-rate alerts (Google SRE):')
for window, burn, action, meaning in [
    ('1h','14.4x','PAGE','budget gone in 1h at this rate'),
    ('6h','6x','TICKET','budget gone this week if sustained'),
    ('1d','3x','Slack alert','trend warning'),
    ('3d','1x','Dashboard','baseline drift, capacity check'),
]:
    print(f'  {window:5s} @ {burn:6s} -> {action:12s} | {meaning}')
print()
print('Always link a runbook URL on the alert -- on-call should not guess.')


## Ex 6: Eval-on-Prod Sampling


In [ ]:
print('Eval-on-prod design:')
import random
calls_per_day = 100_000
for sample_rate, judge_cost_per, daily_samples in [
    (0.001, 0.01, int(100_000 * 0.001)),
    (0.01,  0.01, int(100_000 * 0.01)),
    (0.05,  0.01, int(100_000 * 0.05)),
]:
    daily_cost = daily_samples * judge_cost_per
    print(f'  sample {sample_rate:.1%} -> {daily_samples:5d}/day | judge ${daily_cost:.2f}/day')
print()
print('1% is the common default - large enough to detect drift, small enough to afford.')
print('Cap with a virtual key so judge spend never runs away.')


## Ex 7: PII Redaction


In [ ]:
print('PII redaction layers (before logs):')
for layer, tool, what in [
    ('Built-in recognizers','Presidio','emails, phones, credit cards, IPs'),
    ('Custom recognizers','Presidio + regex','account_id, order_id, PAN card'),
    ('ML-based NER','GCP DLP / presidio transformer','names, orgs, rare formats'),
    ('Short-retention store','S3 + KMS, 7d TTL','full raw content for audit'),
    ('Access controls','IAM + SSO groups','restrict DSAR / audit access'),
]:
    print(f'  {layer:22s}: {tool:30s} | {what}')


---
## Recap
4 golden signals + cost + quality. Langfuse traces + Prometheus metrics + OTel spans + correlation IDs.
Hash prompts by default; PII redact full content + short retention.
Burn-rate alerts (14.4x/1h page, 6x/6h ticket) beat raw thresholds.
Every alert links a runbook. Every prod question answerable in < 60s.
